# Adversarial ASR Demo — Live Gradio Interface

This notebook runs the interactive Gradio demo using pre-trained perturbations.

## Perturbations
- **Targeted CW Injection** — `results/ucw_delta_working.pt`
- **Untargeted UAP** — `results/universal_perturbation_v_80.pt`

## Target Phrase
- "This is a Demo - aai590"


In [1]:
# ── Imports & seeding ──────────────────────────────────────────────────────
import os
import json
import torch
import numpy as np
import soundfile as sf
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gradio as gr
import resampy

torch.manual_seed(42)
np.random.seed(42)

# ── Device setup ───────────────────────────────────────────────────────────
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

from src.models.whisper_wrapper import WhisperASRWithAttack
import src.attacks as attacks

# ── Configuration ──────────────────────────────────────────────────────────
TARGET_PHRASE = "access evil.com"
CW_PERT_PATH  = Path('results/ucw_delta_working.pt')
UAP_PERT_PATH = Path('results/universal_perturbation_v_80.pt')

print(f"CW  perturbation : {CW_PERT_PATH}  (exists: {CW_PERT_PATH.exists()})")
print(f"UAP perturbation : {UAP_PERT_PATH} (exists: {UAP_PERT_PATH.exists()})")


Using device: mps
CW  perturbation : results/ucw_delta_working.pt  (exists: True)
UAP perturbation : results/universal_perturbation_v_80.pt (exists: True)


## Step 6: Live Gradio Demo

An interactive Gradio interface for the presentation. Three modes are available:

| Mode | Description |
|---|---|
| **Clean** | Standard Whisper transcription — baseline reference |
| **Untargeted UAP** | Universal imperceptible noise that degrades ASR accuracy |
| **Targeted CW Injection** | Forces Whisper to output *"This is a Demo - aai590"* regardless of input |

**Prerequisites:** Complete Steps 1–5 so that `demo_assets/targeted_perturbation.pt` exists.  
The UAP (`results/universal_perturbation_v.pt`) is loaded from the earlier training run if available.


In [2]:
# ── Load perturbations ─────────────────────────────────────────────────────
targeted_pert = torch.load(CW_PERT_PATH, map_location='cpu').reshape(-1)
uap           = torch.load(UAP_PERT_PATH, map_location='cpu').reshape(-1)

print(f"CW  perturbation loaded — {targeted_pert.shape[0]} samples ({targeted_pert.shape[0]/16000:.2f}s)")
print(f"UAP perturbation loaded — {uap.shape[0]} samples ({uap.shape[0]/16000:.2f}s)")

# ── Load model ─────────────────────────────────────────────────────────────
model = WhisperASRWithAttack(model_path="openai/whisper-base", device=device)
print("Whisper model loaded.")


CW  perturbation loaded — 160000 samples (10.00s)
UAP perturbation loaded — 80000 samples (5.00s)


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Whisper model loaded.


In [3]:
# ── Audio utilities ────────────────────────────────────────────────────────

def preprocess_audio(audio_tuple):
    """Convert Gradio (sr, ndarray) → float32 numpy array at 16 kHz."""
    sr, data = audio_tuple
    if data.dtype == np.int16:
        data = data.astype(np.float32) / 32768.0
    elif data.dtype != np.float32:
        data = data.astype(np.float32)
    if data.ndim == 2:                          # stereo → mono
        data = data.mean(axis=1)
    if sr != 16000:
        data = resampy.resample(data, sr, 16000)
    return np.clip(data, -1.0, 1.0)


def apply_perturbation(audio: np.ndarray, pert: torch.Tensor) -> np.ndarray:
    """Tile perturbation to match audio length, add it, and clamp to [-1, 1]."""
    pert_np = pert.numpy()
    if len(pert_np) < len(audio):
        pert_np = np.tile(pert_np, int(np.ceil(len(audio) / len(pert_np))))
    return np.clip(audio + pert_np[: len(audio)], -1.0, 1.0).astype(np.float32)


def compute_snr(orig: np.ndarray, adv: np.ndarray) -> float:
    """Return signal-to-noise ratio (dB) between original and adversarial audio."""
    n = min(len(orig), len(adv))
    sig_pwr   = np.mean(orig[:n] ** 2)
    noise_pwr = np.mean((orig[:n] - adv[:n]) ** 2)
    return float('inf') if noise_pwr < 1e-12 else 10.0 * np.log10(sig_pwr / noise_pwr)


def waveform_comparison_fig(orig: np.ndarray, adv: np.ndarray, adv_label: str) -> plt.Figure:
    """Return a 2-panel figure comparing original and adversarial waveforms."""
    fig, (ax_orig, ax_adv) = plt.subplots(2, 1, figsize=(10, 4), sharex=False)
    for ax, wave, color, title, xlabel in [
        (ax_orig, orig, 'steelblue', 'Original Audio',  ''),
        (ax_adv,  adv,  'crimson',   adv_label,          'Time (s)'),
    ]:
        t = np.linspace(0, len(wave) / 16000, len(wave))
        ax.plot(t, wave, color=color, linewidth=0.4, alpha=0.8)
        ax.set_title(title)
        ax.set_ylabel('Amplitude')
        ax.set_xlabel(xlabel)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig


In [4]:
# ── Inference dispatcher ───────────────────────────────────────────────────

ATTACK_MODES = {
    "Untargeted UAP":        uap,
    "Targeted CW Injection": targeted_pert,
}


def run_inference(audio_input, mode):
    """Transcribe audio with the selected attack mode applied."""
    if audio_input is None:
        return "—", "—", "—", "No audio provided.", None, None

    audio_np   = preprocess_audio(audio_input)
    clean_text = model.transcribe(torch.from_numpy(audio_np))

    if mode == "Clean":
        fig = waveform_comparison_fig(audio_np, audio_np, "No Perturbation Applied")
        return clean_text, clean_text, "N/A", "Clean — no attack", (16000, audio_np.copy()), fig

    pert = ATTACK_MODES.get(mode)
    if pert is None:
        return "—", "—", "—", "Unknown mode.", None, None

    adv_np   = apply_perturbation(audio_np, pert)
    adv_text = model.transcribe(torch.from_numpy(adv_np))
    snr_val  = compute_snr(audio_np, adv_np)
    snr_str  = f"{snr_val:.2f} dB"
    fig      = waveform_comparison_fig(audio_np, adv_np, f"{mode} (SNR {snr_val:.1f} dB)")

    if mode == "Untargeted UAP":
        status = "Untargeted UAP applied"
    else:
        hit    = TARGET_PHRASE.lower() in adv_text.lower()
        status = f"{'✓ Injected' if hit else '✗ Not injected'} — target: \"{TARGET_PHRASE}\""

    return clean_text, adv_text, snr_str, status, (16000, adv_np), fig


In [5]:
# ── Gradio UI ──────────────────────────────────────────────────────────────

SUBTITLE = (
    "Upload **or** record audio, select an attack mode, then click **Run**.\n\n"
    "| Mode | Description |\n"
    "|---|---|\n"
    "| **Clean** | Baseline Whisper transcription — no modification |\n"
    "| **Untargeted UAP** | Imperceptible universal noise that degrades accuracy |\n"
    f"| **Targeted CW Injection** | Forces Whisper to transcribe *\"{TARGET_PHRASE}\"* |\n"
)

with gr.Blocks(
    title="SoundFinal — Adversarial ASR Demo (AAI-590)",
    theme=gr.themes.Soft(primary_hue="blue"),
) as demo_app:

    gr.Markdown("# SoundFinal — Adversarial Speech Attack Demo  \n### AAI-590 Final Project")
    gr.Markdown(SUBTITLE)

    with gr.Row():
        with gr.Column(scale=1, min_width=300):
            audio_in = gr.Audio(
                sources=["microphone", "upload"],
                type="numpy",
                label="Input Audio (record or upload a WAV/MP3)",
            )
            mode_sel = gr.Radio(
                choices=["Clean"] + list(ATTACK_MODES.keys()),
                value="Clean",
                label="Attack Mode",
            )
            run_btn = gr.Button("▶  Run", variant="primary", size="lg")

        with gr.Column(scale=2):
            with gr.Row():
                clean_box = gr.Textbox(label="Clean Transcript",       interactive=False, lines=2)
                adv_box   = gr.Textbox(label="Adversarial Transcript", interactive=False, lines=2)
            with gr.Row():
                snr_box    = gr.Textbox(label="SNR",    interactive=False, scale=1)
                status_box = gr.Textbox(label="Status", interactive=False, scale=3)
            adv_audio_out = gr.Audio(label="Adversarial Audio (playback)", type="numpy")
            waveform_plot = gr.Plot(label="Waveform Comparison")

    run_btn.click(
        fn=run_inference,
        inputs=[audio_in, mode_sel],
        outputs=[clean_box, adv_box, snr_box, status_box, adv_audio_out, waveform_plot],
    )

    gr.Markdown(
        "> **Tip:** Using a microphone on macOS requires mic access in "
        "*System Settings → Privacy & Security → Microphone*."
    )

demo_app.launch(share=False, inbrowser=True, quiet=False)


/var/folders/z0/k0hwwqsn2rq7f2kbhjp_5lzw0000gn/T/ipykernel_78463/4125422845.py:12: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


/opt/homebrew/anaconda3/envs/capstone/lib/python3.11/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [1, 3001, 201]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it w

## Troubleshooting Tips

1. **Attack fails (target phrase not found)**: Increase `CW_C` (e.g., `100.0`) or `CW_STEPS` (e.g., `1000`). A larger `c` puts more weight on the adversarial loss relative to the L2 penalty.
2. **Low SNR / audible artifacts**: Reduce `CW_C` — the binary search (`CW_BS_STEPS`) will find a tighter `c` automatically. Also ensure `CW_STEPS >= 500`.
3. **Out of memory (MPS/CUDA)**: Reduce to `model_path="openai/whisper-tiny"`. All `WhisperASRWithAttack` calls support any whisper variant.
4. **Recording is silent / near-zero**: Ensure your microphone is permitted in macOS System Settings → Privacy → Microphone. Check that `sounddevice` lists your expected device with `import sounddevice as sd; print(sd.query_devices())`.
5. **`VOICE_FILE` missing on model step**: Run Step 1 (recording cell) first, or place a 16 kHz WAV at `demo_assets/my_voice_generic.wav` manually.
